# 00 — Preflight and upstream Stage-0 diagnosis

This repair-first run withdraws the earlier scientific verdict until the
causal instrument passes calibration. Stage 0 first records the required
environment checks, then executes the released Jacobian Lens walkthrough's
model/lens/readout cells with byte-identical source. The public walkthrough
contains no causal intervention cell; readout success is therefore reported
separately from the unavailable canonical swap.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path('/home/jovyan/j-space-thoughts')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['HF_HOME'] = str(Path.home() / '.cache/huggingface')
os.environ['HF_HUB_CACHE'] = str(Path.home() / '.cache/huggingface/hub')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(Path.home() / '.cache/huggingface/hub')

from src.v2_stage0 import collect_preflight, print_preflight

preflight = collect_preflight()
print_preflight(preflight)
assert preflight['status'] == 'PASS', preflight

PATH=/home/jovyan/.local/bin:/home/jovyan/.npm-global/bin:/home/jovyan/.local/bin:/home/jovyan/.npm-global/bin:/command:/opt/rsync/usr/bin:/opt/ssh/usr/bin:/opt/ssh/usr/sbin:/opt/sudo/usr/bin:/opt/sudo/usr/sbin:/opt/conda/bin:/opt/conda/condabin:/opt/conda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin
codex: /home/jovyan/.npm-global/bin/codex
hf: /home/jovyan/.local/bin/hf
git: /usr/bin/git
nvidia-smi: /usr/bin/nvidia-smi

[codex_version] returncode=0
codex-cli 0.142.5

[hf_auth] returncode=0
user=sushmanth orgs=devoworm-group,sushmanthreddy,OWG,syscv-community,context-course

[git_global_config] returncode=0
user.name=sushmanthreddy
user.email=sushmanthreddymereddy@cisco.com
init.defaultbranch=main
credential.helper=store --file /home/jovyan/.git-credentials

[git_remote] returncode=0
origin	https://github.com/sushmanthreddy/j-space-thoughts.git (fetch)
origin	https://github.com/sushmanthreddy/j-space-thoughts.git (push)

[gpu] returncode=0
name, memory.total [MiB],

## Released walkthrough readout (unchanged source)

The next four code cells are exact copies of cells 1, 3, 5, and 7 from the
pinned upstream `walkthrough.ipynb`. They demonstrate loading and readout only.
No activation mutation or swapped continuation is present in the release.

In [2]:
import jlens

jlens.configure_logging()

MODEL_NAME = "Qwen/Qwen3.5-4B"
# MODEL_NAME = "Qwen/Qwen3.6-27B"

LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "qwen-n1000"
LENS_FILE = {
    "Qwen/Qwen3.5-4B": "qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
    "Qwen/Qwen3.6-27B": "qwen3.6-27b/jlens/Salesforce-wikitext/Qwen3.6-27B_jacobian_lens_n1000.pt",
}[MODEL_NAME]

In [3]:
import torch
import transformers

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16
).cuda()
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)
model

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

HFLensModel(Qwen3_5ForCausalLM, n_layers=32, d_model=2560)

In [4]:
lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename=LENS_FILE, revision=LENS_REVISION
)
lens

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

JacobianLens(d_model=2560, n_prompts=1000, source_layers=[0..30] (31 layers))

In [5]:
prompt = "Fact: The currency used in the country shaped like a boot is"
layers = [
    model.n_layers // 4,
    model.n_layers // 2,
    model.n_layers // 4 * 3,
    model.n_layers - 2,
]

jlens_logits, model_logits, _ = lens.apply(model, prompt, layers=layers, positions=[-2])
logit_lens, _, _ = lens.apply(
    model, prompt, layers=layers, positions=[-2], use_jacobian=False
)


def top5(logits):
    return [tokenizer.decode([t]) for t in logits.topk(5).indices]


for layer in layers:
    print(f"L{layer:>3} logit-lens: {top5(logit_lens[layer][0])}")
    print(f"L{layer:>3} J-lens:     {top5(jlens_logits[layer][0])}")
print(f"model:           {top5(model_logits[0])}")

L  8 logit-lens: ['oman', 'edom', 'ולי', ' Urlaubs', 'GPC']
L  8 J-lens:     [' `', ' boots', ' *', ' `\\', ' heel']
L 16 logit-lens: ['shaw', 'วย', 'amaz', 'REA', '举世']
L 16 J-lens:     ['?', '？', "'?", ' Italy', '____']
L 24 logit-lens: ['的形状', '形状的', 'shape', '形状', '-shaped']
L 24 J-lens:     ['-shaped', ' shape', ' shaped', 'shape', '形状']
L 30 logit-lens: [' is', ' shape', '-shaped', ' heel', ' shaped']
L 30 J-lens:     [' is', ' shape', ' shaped', '-shaped', ' heel']
model:           [' is', ' in', ' on', '.', ' with']


## Release audit and Stage-0 decision

The audit below verifies dependency/file hashes, scans the released
walkthrough/API/tests, persists the readout output, and creates F0. Missing
upstream causal code is classified as a release omission—not as a Qwen model
failure and not as a successful swap.

In [6]:
import json

from src.v2_stage0 import (
    audit_upstream_release,
    collect_upstream_readout,
    persist_stage0,
)

upstream_audit = audit_upstream_release()
upstream_readout = collect_upstream_readout(
    tokenizer=tokenizer,
    jlens_logits=jlens_logits,
    logit_lens=logit_lens,
    model_logits=model_logits,
    layers=layers,
    model_name=MODEL_NAME,
    lens_repo=LENS_REPO,
    lens_revision=LENS_REVISION,
    lens_file=LENS_FILE,
    model_commit=getattr(hf_model.config, '_commit_hash', None),
)
repair = persist_stage0(
    preflight=preflight,
    upstream_audit=upstream_audit,
    upstream_readout=upstream_readout,
)
print(json.dumps({
    'stage0': repair['stage0']['status'],
    'decision': repair['stage0']['decision'],
    'upstream_readout': repair['gate_ledger']['upstream_readout'],
    'upstream_causal_swap': repair['gate_ledger']['upstream_causal_swap'],
    'g_swap': repair['gate_ledger']['g_swap'],
    'science_allowed': repair['stage0']['science_allowed'],
}, indent=2))
assert repair['gate_ledger']['g_swap'] == 'UNTESTED'
assert repair['stage0']['science_allowed'] is False

{
  "stage0": "COMPLETE_WITH_RELEASE_OMISSION",
  "decision": "UPSTREAM_CAUSAL_SWAP_NOT_RUNNABLE_RELEASE_OMISSION",
  "upstream_readout": "PASS",
  "upstream_causal_swap": "NOT_RUNNABLE_RELEASE_OMISSION",
  "g_swap": "UNTESTED",
  "science_allowed": false
}


In [7]:
import gc

del lens, model, tokenizer, hf_model
gc.collect()
torch.cuda.empty_cache()
print('Stage 0 complete: proceed only to Stage 1 custom repair; Stage 2/3 remain gated.')

Stage 0 complete: proceed only to Stage 1 custom repair; Stage 2/3 remain gated.
